# Optimizing Large DataFrames with Chunking

### Table of Contents
<a id="toc"></a>
1. [Introduction](#introduction)
2. [Set up](#set-up)
3. [Chunk Creation and Exploration](#chunk-exploration)
4. [Chunk Data Type Consistency](#chunk-consistency)
5. [Column Data Type Conversions](#column-conversions)
6. [Downcasting Experiment](#downcasting)
7. [Conclusion](#conclusion)

### Introduction
<a id="introduction"></a>
In this guided project, we will demonstrate how to chunk DataFrames and optimize memory usage. We are working with financial data from [Lending Club](https://www.lendingclub.com/), a company that manages peer-to-peer connections between borrowers and investors. While the original dataset is no longer hosted on the Lending Club website, it is available [on Kaggle](https://www.kaggle.com/datasets/wordsforthewise/lending-club/data). The Kaggle version covers 2007 to 2018; however, our dataset has been specifically tailored for this project to include data from 2007 to 2011.

To simulate a constrained environment, we will operate under the assumption that we only have **10 MB** of available memory. This constraint allows us to establish a clear "benchmark" for chunking our DataFrames effectively.

### Setup
<a id="set-up"></a>
We will begin by importing the necessary libraries and inspecting the first five rows of our dataset.

[Back to Table of Contents](#toc)

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
pd.options.display.max_columns = 99

In [2]:
first_five = pd.read_csv('loans_2007.csv', nrows=5)
first_five

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599.0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,NaN,10+ years,RENT,24000.0,Verified,Dec-2011,Fully Paid,n,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.00,0.00,5863.155187,5833.84,5000.00,863.16,0.00,0.00,0.00,Jan-2015,171.62,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,1077430,1314167.0,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,Ryder,< 1 year,RENT,30000.0,Source Verified,Dec-2011,Charged Off,n,car,bike,309xx,GA,1.00,0.0,Apr-1999,5.0,3.0,0.0,1687.0,9.4%,4.0,f,0.00,0.00,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,Apr-2013,119.66,Sep-2013,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,1077175,1313524.0,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,NaN,10+ years,RENT,12252.0,Not Verified,Dec-2011,Fully Paid,n,small_business,real estate business,606xx,IL,8.72,0.0,Nov-2001,2.0,2.0,0.0,2956.0,98.5%,10.0,f,0.00,0.00,3005.666844,3005.67,2400.00,605.67,0.00,0.00,0.00,Jun-2014,649.91,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
3,1076863,1277178.0,10000.0,10000.0,10000.0,36 months,13.49%,339.31,C,C1,AIR RESOURCES BOARD,10+ years,RENT,49200.0,Source Verified,Dec-2011,Fully Paid,n,other,personel,917xx,CA,20.00,0.0,Feb-1996,1.0,10.0,0.0,5598.0,21%,37.0,f,0.00,0.00,12231.890000,12231.89,10000.00,2214.92,16.97,0.00,0.00,Jan-2015,357.48,Apr-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
4,1075358,1311748.0,3000.0,3000.0,3000.0,60 months,12.69%,67.79,B,B5,University Medical Group,1 year,RENT,80000.0,Source Verified,Dec-2011,Current,n,other,Personal,972xx,OR,17.94,0.0,Jan-1996,0.0,15.0,0.0,27783.0,53.9%,38.0,f,461.73,461.73,3581.120000,3581.12,2538.27,1042.85,0.00,0.00,0.00,Jun-2016,67.79,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0


### Chunk Creation and Exploration
<a id="chunk-exploration"></a>
With the environment set up and our initial data inspection complete, we need to address memory usage. To understand the footprint of our data, we will start by reading in a sample of 1,000 rows and calculating the memory consumption.

In [3]:
thousand_chunk = pd.read_csv('loans_2007.csv', nrows=1000)
print(thousand_chunk.memory_usage(deep=True).sum()/(1024*1024))

1.3676795959472656


The sample of 1,000 rows occupies only 1.4 MB of memory. Based on this, we can safely increase our chunk size to 3,000 rows. Let's examine the memory footprint across all 3,000-row chunks.

[Back to Table of Contents](#toc)

In [4]:
chunk_iter = pd.read_csv('loans_2007.csv', chunksize=3000)
for chunk in chunk_iter:
    print(chunk.memory_usage(deep=True).sum()/(1024*1024))

4.1016998291015625
4.098297119140625
4.099673271179199
4.1008806228637695
4.097287178039551
4.0987653732299805
4.097776412963867
4.09971809387207
4.097790718078613
4.097665786743164
4.110683441162109
4.10931396484375
4.115175247192383
4.326666831970215
0.7793684005737305


Now that we have defined our chunks, we want to analyze the specifics of each one to identify optimization opportunities. We will focus on:

* The data type (dtype) of each column.
* The number of unique values per column.
* The count of missing (null) values.
* The percentage of values that are unique.
* Float columns with no missing values that are candidates for conversion to integers.
* A sample of the first four values in each column to verify data formatting.

In [5]:
def profile_chunk(chunk):
    # Memory Footprint (in Megabytes)
    memory_mb = chunk.memory_usage(deep=True).sum() / (1024**2)

    profile_data = []

    for col in chunk.columns:
        # Basic stats
        dtype = chunk[col].dtype
        num_unique = chunk[col].nunique()
        total_rows = len(chunk)
        unique_pct = (num_unique / total_rows) * 100
        
        # Missing values check
        has_missing = chunk[col].isnull().any()
        missing_label = "Yes" if has_missing else "No"
        
        # Get the first 4 values as a comma-separated string
        samples = ", ".join(chunk[col].head(4).astype(str).tolist())
        
        # Integer Candidate check
        is_int_candidate = "No"
        if pd.api.types.is_float_dtype(dtype) and not has_missing:
            if (chunk[col] == chunk[col].apply(int)).all():
                is_int_candidate = "Yes"

        profile_data.append({
            "Column": col,
            "Dtype": dtype,
            "Uniq_Count": num_unique,
            "Uniq_Pct": f"{unique_pct:.2f}%",
            "Has_Missing": missing_label,
            "Int_Candidate": is_int_candidate,
            "Sample_Values": samples
        })

    # Create the summary DataFrame
    summary_df = pd.DataFrame(profile_data)
    
    return summary_df, memory_mb

In [6]:
chunk_iter = pd.read_csv('loans_2007.csv', chunksize=3000)
chunk_label = 1
total_memory = 0
for chunk in chunk_iter:
    print(f"Chunk {chunk_label}")
    chunk_summary, chunk_footprint = profile_chunk(chunk)
    print(f"Memory Footprint: {chunk_footprint:.4f} MB")
    
    display(chunk_summary) 

    chunk_label += 1
    total_memory += chunk_footprint
    print("\n")
    print("-"*85, "\n")

Chunk 1
Memory Footprint: 4.1017 MB


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,int64,3000,100.00%,No,No,"1077501, 1077430, 1077175, 1076863"
1,member_id,float64,3000,100.00%,No,Yes,"1296599.0, 1314167.0, 1313524.0, 1277178.0"
2,loan_amnt,float64,390,13.00%,No,Yes,"5000.0, 2500.0, 2400.0, 10000.0"
3,funded_amnt,float64,438,14.60%,No,Yes,"5000.0, 2500.0, 2400.0, 10000.0"
4,funded_amnt_inv,float64,663,22.10%,No,No,"4975.0, 2500.0, 2400.0, 10000.0"
5,term,object,2,0.07%,No,No,"36 months, 60 months, 36 months, 36 months"
6,int_rate,object,36,1.20%,No,No,"10.65%, 15.27%, 15.96%, 13.49%"
7,installment,float64,1591,53.03%,No,No,"162.87, 59.83, 84.33, 339.31"
8,grade,object,7,0.23%,No,No,"B, C, C, C"
9,sub_grade,object,35,1.17%,No,No,"B2, C4, C5, C1"




------------------------------------------------------------------------------------- 

Chunk 2
Memory Footprint: 4.0983 MB


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,int64,3000,100.00%,No,No,"1026242, 1027822, 1022057, 1027992"
1,member_id,float64,3000,100.00%,No,Yes,"1255399.0, 1257204.0, 1250835.0, 1257369.0"
2,loan_amnt,float64,397,13.23%,No,Yes,"35000.0, 1800.0, 16000.0, 3000.0"
3,funded_amnt,float64,397,13.23%,No,Yes,"35000.0, 1800.0, 16000.0, 3000.0"
4,funded_amnt_inv,float64,667,22.23%,No,No,"34647.35244739579, 1800.0, 16000.0, 3000.0"
5,term,object,2,0.07%,No,No,"60 months, 36 months, 36 months, 36 months"
6,int_rate,object,36,1.20%,No,No,"17.27%, 7.51%, 17.27%, 12.42%"
7,installment,float64,1541,51.37%,No,No,"874.93, 56.0, 572.6, 100.25"
8,grade,object,7,0.23%,No,No,"D, A, D, B"
9,sub_grade,object,35,1.17%,No,No,"D3, A3, D3, B4"




------------------------------------------------------------------------------------- 

Chunk 3
Memory Footprint: 4.0997 MB


...



------------------------------------------------------------------------------------- 

Chunk 14
Memory Footprint: 4.3267 MB


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,object,3000,100.00%,No,No,"298963, 298946, 298649, 297158"
1,member_id,float64,2999,99.97%,Yes,No,"298960.0, 298943.0, 298568.0, 297155.0"
2,loan_amnt,float64,315,10.50%,Yes,No,"18500.0, 6000.0, 10000.0, 10000.0"
3,funded_amnt,float64,361,12.03%,Yes,No,"18500.0, 6000.0, 10000.0, 10000.0"
4,funded_amnt_inv,float64,1655,55.17%,Yes,No,"7082.589999999998, 4600.0, 9003.43, 8604.26"
5,term,object,2,0.07%,Yes,No,"36 months, 36 months, 36 months, 36 months"
6,int_rate,object,262,8.73%,Yes,No,"10.71%, 12.92%, 11.66%, 11.34%"
7,installment,float64,2400,80.00%,Yes,No,"603.13, 201.94, 330.53, 329.0"
8,grade,object,7,0.23%,Yes,No,"B, D, C, C"
9,sub_grade,object,35,1.17%,Yes,No,"B5, D2, C3, C2"




------------------------------------------------------------------------------------- 

Chunk 15
Memory Footprint: 0.7794 MB


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,object,538,100.00%,No,No,"247286, 246996, 246720, 246535"
1,member_id,float64,536,99.63%,Yes,No,"247257.0, 244258.0, 246706.0, 246427.0"
2,loan_amnt,float64,149,27.70%,Yes,No,"6000.0, 17250.0, 13000.0, 12000.0"
3,funded_amnt,float64,150,27.88%,Yes,No,"6000.0, 17250.0, 13000.0, 12000.0"
4,funded_amnt_inv,float64,354,65.80%,Yes,No,"4201.94, 12150.005316292236, 7700.0, 5650.0"
5,term,object,1,0.19%,Yes,No,"36 months, 36 months, 36 months, 36 months"
6,int_rate,object,86,15.99%,Yes,No,"11.34%, 17.66%, 15.13%, 18.29%"
7,installment,float64,469,87.17%,Yes,No,"197.4, 620.7, 451.48, 435.58"
8,grade,object,7,1.30%,Yes,No,"C, G, E, G"
9,sub_grade,object,35,6.51%,Yes,No,"C2, G2, E4, G4"




------------------------------------------------------------------------------------- 



[Back to Table of Contents](#toc)

We now have a comprehensive summary of each chunk. Next, we will calculate the total memory consumption of the entire dataset by aggregating the memory usage of all combined chunks.

In [7]:
print(f"Total Memory: {total_memory}")

Total Memory: 58.43076229095459


### Chunk Data Type Consistency
<a id="chunk-consistency"></a>
Next, we will differentiate between our string (object) columns and our numeric columns.

In [8]:
chunk_iter = pd.read_csv('loans_2007.csv', chunksize=3000)

summaries = []

for chunk in chunk_iter:
    summary, footprint = profile_chunk(chunk)
    summaries.append(summary)

In [9]:
# baseline from the first chunk
first_chunk = summaries[0]

numeric_cols = first_chunk[first_chunk['Dtype'].astype(str).str.contains('int|float')]['Column'].tolist()
string_cols = first_chunk[first_chunk['Dtype'].astype(str).str.contains('object')]['Column'].tolist()

print(f"Numeric Columns ({len(numeric_cols)}): {numeric_cols}\n")
print(f"String Columns ({len(string_cols)}): {string_cols}")

Numeric Columns (31): ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'total_acc', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt', 'collections_12_mths_ex_med', 'policy_code', 'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt', 'pub_rec_bankruptcies', 'tax_liens']

String Columns (21): ['term', 'int_rate', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'revol_util', 'initial_list_status', 'last_pymnt_d', 'last_credit_pull_d', 'application_type']


Now that we have identified the column types, we need to verify consistency across the dataset. We will check if the data types remain uniform across all chunks or if any columns were assigned different types due to varying data in specific chunks.

In [10]:
dtype_changes = []

# Column/dtype mapping from the first chunk to compare against
baseline_dtypes = summaries[0].set_index('Column')['Dtype']

for i, report in enumerate(summaries):
    current_dtypes = report.set_index('Column')['Dtype']
    
    # Check if any dtypes in the current chunk differ from our baseline
    inconsistent = current_dtypes[current_dtypes != baseline_dtypes]
    
    if not inconsistent.empty:
        for col, dtype in inconsistent.items():
            dtype_changes.append({
                "Chunk": i + 1,
                "Column": col,
                "Original_Dtype": baseline_dtypes[col],
                "New_Dtype": dtype
            })

if not dtype_changes:
    print("Consistency Check: PASS. All columns have consistent types across all 15 chunks.")
else:
    print("Consistency Check: FAIL. Found the following type shifts:")
    display(pd.DataFrame(dtype_changes))

Consistency Check: FAIL. Found the following type shifts:


,Chunk,Column,Original_Dtype,New_Dtype
0,14,id,int64,object
1,15,id,int64,object


Our analysis shows that the data contains 31 numeric columns and 21 string columns. We discovered that in chunks 14 and 15, the `id` column was assigned as an object/string type, whereas previous chunks identified it as an integer. Since this column will not be used for calculations or modeling, we will leave it as is for now.

Let's begin optimizing our string (object) columns. Our goal is to convert these to more efficient types, such as `category`, `float`, or `datetime`. We will apply the following logic:

* **Object columns** with less than 50% unique values will be converted to the `category` type.
* **Object columns** containing numeric data will be converted to `float`.

The code below inspects the object columns from the first chunk to identify specific candidates for conversion.

[Back to Table of Contents](#toc)

### Column Data Type Conversions
<a id="column-conversions"></a>

In [11]:
first_chunk[first_chunk['Dtype'] == 'object']

,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
5,term,object,2,0.07%,No,No,"36 months, 60 months, 36 months, 36 months"
6,int_rate,object,36,1.20%,No,No,"10.65%, 15.27%, 15.96%, 13.49%"
8,grade,object,7,0.23%,No,No,"B, C, C, C"
9,sub_grade,object,35,1.17%,No,No,"B2, C4, C5, C1"
10,emp_title,object,2653,88.43%,Yes,No,"nan, Ryder, nan, AIR RESOURCES BOARD"
11,emp_length,object,11,0.37%,Yes,No,"10+ years, < 1 year, 10+ years, 10+ years"
12,home_ownership,object,3,0.10%,No,No,"RENT, RENT, RENT, RENT"
14,verification_status,object,3,0.10%,No,No,"Verified, Source Verified, Not Verified, Sourc..."
15,issue_d,object,2,0.07%,No,No,"Dec-2011, Dec-2011, Dec-2011, Dec-2011"
16,loan_status,object,6,0.20%,No,No,"Fully Paid, Charged Off, Fully Paid, Fully Paid"


Based on our inspection, we have decided to implement the following data type changes:

* `id`: **`object`**
* `term`, `grade`, `sub_grade`, `emp_length`, `home_ownership`, `verification_status`, `loan_status`, `pymnt_plan`, `purpose`, `title`, `zip_code`, `addr_state`, `initial_list_status`, `application_type`: **`category`**
* `int_rate`, `revol_util`: **`float`**
* `issue_d`, `earliest_cr_line`, `last_pymnt_d`, `last_credit_pull_d`: **`datetime`**

In [12]:
improved_col_dtypes = {
    'id': 'object', 'term': 'category', 'grade': 'category', 'sub_grade': 'category',
    'emp_length': 'category', 'home_ownership': 'category',
    'verification_status': 'category', 'loan_status': 'category', 
    'pymnt_plan': 'category', 'purpose': 'category', 'title': 'category', 
    'zip_code': 'category', 'addr_state': 'category', 
    'initial_list_status': 'category', 'application_type': 'category'
}

In [13]:
chunk_iter = pd.read_csv(
    'loans_2007.csv', 
    chunksize=3000, 
    dtype=improved_col_dtypes, 
    parse_dates=["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"],
    date_format='%b-%Y'
)

summaries = []
total_memory = 0

for chunk in chunk_iter:
    summary, footprint = profile_chunk(chunk)
    summaries.append(summary)
    total_memory += footprint

print(f"Total Memory Across All Chunks: {total_memory:.2f} MB")
print("\n")
print("First Chunk:")
display(summaries[0])

Total Memory Across All Chunks: 24.31 MB


First Chunk:


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,object,3000,100.00%,No,No,"1077501, 1077430, 1077175, 1076863"
1,member_id,float64,3000,100.00%,No,Yes,"1296599.0, 1314167.0, 1313524.0, 1277178.0"
2,loan_amnt,float64,390,13.00%,No,Yes,"5000.0, 2500.0, 2400.0, 10000.0"
3,funded_amnt,float64,438,14.60%,No,Yes,"5000.0, 2500.0, 2400.0, 10000.0"
4,funded_amnt_inv,float64,663,22.10%,No,No,"4975.0, 2500.0, 2400.0, 10000.0"
5,term,category,2,0.07%,No,No,"36 months, 60 months, 36 months, 36 months"
6,int_rate,object,36,1.20%,No,No,"10.65%, 15.27%, 15.96%, 13.49%"
7,installment,float64,1591,53.03%,No,No,"162.87, 59.83, 84.33, 339.31"
8,grade,category,7,0.23%,No,No,"B, C, C, C"
9,sub_grade,category,35,1.17%,No,No,"B2, C4, C5, C1"


[Back to Table of Contents](#toc)

By simply reassigning the data types for the majority of our object columns, we have reduced the total memory usage from **58.4 MB to 24.3 MB**. This is a significant improvement! We will now address the remaining numeric columns and refine the object columns that require more specific cleaning.

In [14]:
chunk_iter = pd.read_csv(
    'loans_2007.csv', 
    chunksize=3000, 
    dtype=improved_col_dtypes, 
    parse_dates=["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"],
    date_format='%b-%Y'
)

summaries = []
optimized_chunks = []
total_memory = 0

for chunk in chunk_iter:
    # Cleaning Specific Columns ('term', 'int_rate', 'revol_util')
    if 'term' in chunk.columns:
        term_cleaned = chunk['term'].str.strip().str.replace(' months', '')
        chunk['term'] = pd.to_numeric(term_cleaned, errors='coerce').astype('Int64')
        
    # int_rate and revol_util: Strip % and convert to float
    for col in ['int_rate', 'revol_util']:
        if col in chunk.columns and chunk[col].dtype == 'object':
            chunk[col] = chunk[col].str.rstrip('%').astype(float)

    # Logic-based Numeric Downcasting
    summary_df, _ = profile_chunk(chunk)
    
    for _, row in summary_df.iterrows():
        col = row['Column']
        dtype = str(row['Dtype'])
        has_missing = row['Has_Missing'] == 'Yes'
        is_int_candidate = row['Int_Candidate'] == 'Yes'
        
        if 'float' in dtype and has_missing:
            chunk[col] = chunk[col].astype('float32')
            
        elif is_int_candidate:
            chunk[col] = chunk[col].astype('int64')

    summary, footprint = profile_chunk(chunk)
    summaries.append(summary)
    total_memory += footprint
    optimized_chunks.append(chunk)

print(f"Optimization Complete!")
print(f"Total Memory Across Optimized Chunks: {total_memory:.2f} MB")
print("\n")
print("First Chunk:")
display(summaries[0])

Optimization Complete!
Total Memory Across Optimized Chunks: 20.27 MB


First Chunk:


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,object,3000,100.00%,No,No,"1077501, 1077430, 1077175, 1076863"
1,member_id,int64,3000,100.00%,No,No,"1296599, 1314167, 1313524, 1277178"
2,loan_amnt,int64,390,13.00%,No,No,"5000, 2500, 2400, 10000"
3,funded_amnt,int64,438,14.60%,No,No,"5000, 2500, 2400, 10000"
4,funded_amnt_inv,float64,663,22.10%,No,No,"4975.0, 2500.0, 2400.0, 10000.0"
5,term,Int64,2,0.07%,No,No,"36, 60, 36, 36"
6,int_rate,float64,36,1.20%,No,No,"10.65, 15.27, 15.96, 13.49"
7,installment,float64,1591,53.03%,No,No,"162.87, 59.83, 84.33, 339.31"
8,grade,category,7,0.23%,No,No,"B, C, C, C"
9,sub_grade,category,35,1.17%,No,No,"B2, C4, C5, C1"


After further adjustments, our memory usage dropped from **24.3 MB to 20.3 MB**. While this decrease is less dramatic than the first round of optimization, every kilobyte counts in a memory-constrained environment. We have successfully made the dataset much more manageable.

[Back to Table of Contents](#toc)

### Downcasting Experiment
<a id="downcasting"></a>
As a final experiment, we will attempt to "downcast" every column in each chunk to its smallest possible subtype based on its actual values. While this may not always be practical in a production environment—as it can lead to inconsistent data types across different chunks—it serves as a great demonstration of the absolute minimum footprint we can achieve.

In [15]:
chunk_iter = pd.read_csv(
    'loans_2007.csv', 
    chunksize=3000, 
    dtype=improved_col_dtypes, 
    parse_dates=["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"],
    date_format='%b-%Y'
)

summaries = []
optimized_chunks = []
total_memory = 0

for chunk in chunk_iter:
    # Cleaning Specific Columns ('term', 'int_rate', 'revol_util')
    if 'term' in chunk.columns:
        term_cleaned = chunk['term'].str.strip().str.replace(' months', '')
        chunk['term'] = pd.to_numeric(term_cleaned, errors='coerce').astype('Int64')
        
    # int_rate and revol_util: Strip % and convert to float
    for col in ['int_rate', 'revol_util']:
        if col in chunk.columns and chunk[col].dtype == 'object':
            chunk[col] = chunk[col].str.rstrip('%').astype(float)

    # Logic-based Numeric Downcasting
    summary_df, _ = profile_chunk(chunk)
    
    for _, row in summary_df.iterrows():
        col = row['Column']
        dtype = str(row['Dtype'])
        has_missing = row['Has_Missing'] == 'Yes'
        is_int_candidate = row['Int_Candidate'] == 'Yes'
        
        if 'float' in dtype and has_missing:
            chunk[col] = chunk[col].astype('float32')
            
        elif is_int_candidate:
            chunk[col] = chunk[col].astype('int64')

    # Downcasting Experiment    
    float_cols = chunk.select_dtypes(include=['float']).columns
    int_cols = chunk.select_dtypes(include=['integer']).columns

    for col in float_cols:
        chunk[col] = pd.to_numeric(chunk[col], downcast='float')
    
    for col in int_cols:
        chunk[col] = pd.to_numeric(chunk[col], downcast='integer')

    summary, footprint = profile_chunk(chunk)
    summaries.append(summary)
    total_memory += footprint
    optimized_chunks.append(chunk)

print(f"Optimization Complete!")
print(f"Total Memory Across Optimized Chunks: {total_memory:.2f} MB")
print("\n")
print("First Chunk:")
display(summaries[0])

Optimization Complete!
Total Memory Across Optimized Chunks: 14.91 MB


First Chunk:


,Column,Dtype,Uniq_Count,Uniq_Pct,Has_Missing,Int_Candidate,Sample_Values
0,id,object,3000,100.00%,No,No,"1077501, 1077430, 1077175, 1076863"
1,member_id,int32,3000,100.00%,No,No,"1296599, 1314167, 1313524, 1277178"
2,loan_amnt,int32,390,13.00%,No,No,"5000, 2500, 2400, 10000"
3,funded_amnt,int32,438,14.60%,No,No,"5000, 2500, 2400, 10000"
4,funded_amnt_inv,float64,663,22.10%,No,No,"4975.0, 2500.0, 2400.0, 10000.0"
5,term,Int8,2,0.07%,No,No,"36, 60, 36, 36"
6,int_rate,float32,36,1.20%,No,No,"10.65, 15.27, 15.96, 13.49"
7,installment,float32,1591,53.03%,No,No,"162.87, 59.83, 84.33, 339.31"
8,grade,category,7,0.23%,No,No,"B, C, C, C"
9,sub_grade,category,35,1.17%,No,No,"B2, C4, C5, C1"


### Conclusion
<a id="conclusion"></a>
Our experiment with downcasting yielded a final memory footprint of **14.91 MB**, down from our initial starting point of nearly 60 MB. Through the strategic use of chunking, converting strings to categorical data, and optimizing numeric types, we successfully reduced our memory usage by approximately 75%. This process demonstrates that even large datasets can be processed on modest hardware if the data types are managed with intent. While the final downcasting step is often avoided in real-world pipelines to maintain type consistency, the overall workflow provides a robust framework for handling data that exceeds available system memory.

[Back to Table of Contents](#toc)